# Graph clustering

**Algorithms:** `compute_k_core`, `compute_label_propagation`, `compute_louvain` from `zarr_vectors_tools.algorithms`

All three operate on the chunked storage layout. `compute_k_core` only needs degree counts (cross-chunk edges contribute correctly). `compute_label_propagation` and `compute_louvain` materialise the full in-memory adjacency once via `build_adjacency` (same pattern as `shortest_path`). Cross-chunk edges currently lack a per-edge weight slot in core, so weighted Louvain on a graph whose heaviest edges straddle chunk boundaries will under-merge — that's a documented limitation, demonstrated below.

1. Build a synthetic planted-partition graph (three K8 cliques, weakly connected)
2. `compute_k_core` — coreness distribution recovers the within-clique structure
3. `compute_label_propagation` — fast unweighted community discovery
4. `compute_louvain` — modularity-optimised, agrees with LPA on this graph
5. Compare the three labellings and Adjusted Rand Index against ground truth
6. Persist Louvain labels back to a new store via `write_graph`

In [1]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import open_zvr
from zarr_vectors.types.graphs import write_graph, read_graph
from zarr_vectors_tools.algorithms import (
    compute_k_core,
    compute_label_propagation,
    compute_louvain,
)

_tmpdir = tempfile.mkdtemp(prefix="zvt_examples_")
STORE         = os.path.join(_tmpdir, "cliques.zarrvectors")
STORE_LABELED = os.path.join(_tmpdir, "cliques_labeled.zarrvectors")
print("Stores:", STORE, STORE_LABELED)

Stores: C:\Users\andre\AppData\Local\Temp\zvt_examples_dgur_ain\cliques.zarrvectors C:\Users\andre\AppData\Local\Temp\zvt_examples_dgur_ain\cliques_labeled.zarrvectors


## 1 · Planted-partition graph

Three K8 cliques placed at well-separated 3D locations; each clique connected to the next by a single bridge edge. Ground-truth community count: 3.

In [2]:
rng = np.random.default_rng(3)
K = 8                                 # clique size
centres = np.array([
    [100., 100., 100.],
    [400., 100., 100.],
    [250., 400., 100.],
], dtype=np.float32)
n_clusters = len(centres)

positions = np.concatenate([
    c + rng.normal(0, 8., (K, 3)).astype(np.float32)
    for c in centres
]).astype(np.float32)

# Intra-clique edges: full K8 within each cluster.
edges = []
for ci in range(n_clusters):
    s = ci * K
    for i in range(K):
        for j in range(i + 1, K):
            edges.append([s + i, s + j])
# Bridge edges: connect clique 0→1, 1→2.
edges.append([K - 1, K])           # clique 0 last → clique 1 first
edges.append([2 * K - 1, 2 * K])   # clique 1 last → clique 2 first
edges = np.asarray(edges, dtype=np.int32)

# Ground-truth labels.
truth = np.repeat(np.arange(n_clusters), K).astype(np.int32)

print(f"nodes        : {len(positions)}")
print(f"edges        : {len(edges)}  ({n_clusters * K * (K - 1) // 2} intra + {n_clusters - 1} bridges)")
print(f"ground truth : {n_clusters} communities of size {K}")

nodes        : 24
edges        : 86  (84 intra + 2 bridges)
ground truth : 3 communities of size 8


In [3]:
write_graph(
    STORE,
    positions=positions,
    edges=edges,
    chunk_shape=(150., 150., 150.),
    bin_shape=(37.5, 37.5, 37.5),
    is_tree=False,
)
store = open_zvr(STORE)
print(f"vertex_count : {store[0].vertex_count}")
print(f"chunk_shape  : {store.chunk_shape}")

vertex_count : 24
chunk_shape  : (150.0, 150.0, 150.0)


## 2 · k-core decomposition

Every vertex inside a K8 has 7 internal neighbours; the bridge nodes have one extra neighbour. The k-core peeling should report coreness 7 for all 24 vertices (the bridge edges are the first thing to peel off, then nothing else can be peeled until we drop a whole K8 at once). `max_core` therefore equals K-1 = 7.

In [4]:
kc = compute_k_core(STORE)
print(f"max_core    : {kc['max_core']}    (expected {K - 1})")
print(f"coreness    : {kc['coreness'].tolist()}")
print(f"core_sizes  : {kc['core_sizes'].tolist()}")

max_core    : 7    (expected 7)
coreness    : [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7]
core_sizes  : [0, 0, 0, 0, 0, 0, 0, 24]


## 3 · Label propagation (LPA)

Fast and unweighted. Synchronous LPA can occasionally oscillate on highly symmetric subgraphs (e.g. K8 where every vertex sees 7 tied labels in round 1) — try a different `seed=` if it doesn't converge. On this graph, `seed=1` converges in ~4 rounds to the three planted communities.

In [5]:
lpa = compute_label_propagation(STORE, max_iter=20, seed=1)
print(f"n_communities  : {lpa['n_communities']}    (expected {n_clusters})")
print(f"converged      : {lpa['converged']}")
print(f"iterations     : {lpa['iterations']}")
print(f"community_sizes: {lpa['community_sizes'].tolist()}")
print(f"labels         : {lpa['labels'].tolist()}")

n_communities  : 3    (expected 3)
converged      : True
iterations     : 4
community_sizes: [8, 8, 8]
labels         : [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2]


## 4 · Louvain modularity optimisation

Returns the same three-community partition on this graph but also reports the modularity `Q`. For three cliques bridged by single edges, `Q` lands close to `(K-1)/(K-1+2/n_clusters)·(1 - 1/n_clusters)`, which for K=8, n_clusters=3 is around 0.6.

In [6]:
lv = compute_louvain(STORE, seed=0)
print(f"n_communities  : {lv['n_communities']}")
print(f"modularity Q   : {lv['modularity']:.4f}")
print(f"iterations     : {lv['iterations']}")
print(f"community_sizes: {lv['community_sizes'].tolist()}")
print(f"labels         : {lv['labels'].tolist()}")

n_communities  : 3
modularity Q   : 0.6434
iterations     : 2
community_sizes: [8, 8, 8]
labels         : [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2]


## 5 · Adjusted Rand Index vs ground truth

ARI = 1 means perfect agreement (up to label permutation); ARI ≈ 0 means agreement no better than chance. Both LPA and Louvain should hit 1.0 on this synthetic graph.

In [7]:
def adjusted_rand_index(a: np.ndarray, b: np.ndarray) -> float:
    """Self-contained ARI (no sklearn dep). a, b are integer label arrays."""
    a = np.asarray(a); b = np.asarray(b)
    n = len(a)
    # Contingency table.
    unique_a, inv_a = np.unique(a, return_inverse=True)
    unique_b, inv_b = np.unique(b, return_inverse=True)
    M = np.zeros((len(unique_a), len(unique_b)), dtype=np.int64)
    for i in range(n):
        M[inv_a[i], inv_b[i]] += 1
    def comb2(x):
        return x * (x - 1) // 2
    sum_comb = comb2(M).sum()
    sum_a = comb2(M.sum(axis=1)).sum()
    sum_b = comb2(M.sum(axis=0)).sum()
    total = comb2(n)
    expected = sum_a * sum_b / total if total else 0.0
    max_index = 0.5 * (sum_a + sum_b)
    if max_index == expected:
        return 1.0
    return (sum_comb - expected) / (max_index - expected)

print(f"ARI(LPA,    truth) = {adjusted_rand_index(lpa['labels'], truth):.4f}")
print(f"ARI(Louvain, truth) = {adjusted_rand_index(lv['labels'],  truth):.4f}")
print(f"ARI(LPA, Louvain)   = {adjusted_rand_index(lpa['labels'], lv['labels']):.4f}")

ARI(LPA,    truth) = 1.0000
ARI(Louvain, truth) = 1.0000
ARI(LPA, Louvain)   = 1.0000


## 6 · Persist Louvain labels in a new store

Same pattern as connected components: `write_back=True` blocks on core *Add 5*, so we re-write the graph with the Louvain labels as a `community` node attribute.

In [ ]:
g = read_graph(STORE)

write_graph(
    STORE_LABELED,
    positions=g["positions"],
    edges=g["edges"],
    chunk_shape=(150., 150., 150.),
    bin_shape=(37.5, 37.5, 37.5),
    is_tree=False,
    node_attributes={
        "community": lv["labels"].astype(np.int32),
        "coreness":  kc["coreness"].astype(np.int32),
    },
)
store_labeled = open_zvr(STORE_LABELED)
comm = store_labeled[0].attributes["community"].compute()
core = store_labeled[0].attributes["coreness"].compute()
print(f"persisted community dtype : {comm.dtype}  shape : {comm.shape}")
print(f"persisted coreness  dtype : {core.dtype}  shape : {core.shape}")
print(f"unique communities in store: {sorted(set(comm.tolist()))}")

persisted community dtype : int32  shape : (24,)
persisted coreness  dtype : int32  shape : (24,)
unique communities in store: [0, 1, 2]


## Summary

| Step | API |
|------|-----|
| k-core decomposition | `compute_k_core(store, level=0)` → `{coreness, max_core, core_sizes}` |
| Label propagation    | `compute_label_propagation(store, max_iter=20, seed=0)` → `{labels, n_communities, iterations, converged, community_sizes}` |
| Louvain              | `compute_louvain(store, weight=None, max_iter=10, seed=0)` → `{labels, modularity, n_communities, iterations, community_sizes}` |
| Persist labels       | Re-run `write_graph(..., node_attributes={...})` |

Known v0 limitations (documented inline in each algorithm's docstring):

- **Cross-chunk edge weights**: cross-chunk edges always contribute unit weight to `compute_louvain(weight=...)` because core has no per-edge weight slot on the cross array. Biases community boundaries away from chunk borders.
- **In-memory adjacency**: LPA and Louvain materialise the full adjacency before iterating (`O(N+E)`). Suitable for graphs that fit in RAM; true streaming requires the per-chunk cross index (catalog Add 2).
- **Dendrogram**: Louvain returns only the final flat labels. To inspect the Phase-2 hierarchy, call once per resolution by post-processing labels at each Louvain inner round.
- **Write-back**: `write_back=True` blocks on core Add 5, just like `compute_connected_components`.